# Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 📊 Phase 1 — Data Understanding

In [ ]:
df = pd.read_csv("C:\\Users\\abhay\\OneDrive\\Desktop\\Data-Analytics-Portfolio\\05_End-To-End-Project\\Healthcare-Readmission-Analysis\\Data\\diabetic_data.csv")

df.head()

The dataset contains hospital encounter records with 50 variables describing patient demographics, admission details, medical treatments, and readmission status. Age is provided in categorical ranges, and several columns contain missing values represented as "?".

In [ ]:
print(df.info())
print(df.shape)

## 🚀 data cleaning.

In [ ]:
# 1️⃣ Check Missing Values
print(df.isnull().sum().sort_values(ascending=False))
# Columns such as weight, max_glu_serum, and A1Cresult contain more than 80% missing values and therefore provide insufficient information for reliable analysis. 

# 2️⃣ Check Duplicate Rows
print(df.duplicated().sum())

# 3️⃣ Unique Patients
print(df["patient_nbr"].nunique())

# 4️⃣ Target Variable Distribution
print(df["readmitted"].value_counts())

# replace ? with NaN value 
df.replace("?",np.nan,inplace=True)

##  Column Selection (For analysis)

In [ ]:
# drop the colume that we are not using in analysis
df = df.drop(
[
"encounter_id",
"weight",
"max_glu_serum",
"A1Cresult",
"payer_code"
],
axis=1
)

In [ ]:
missing_percentage = df.isnull().mean()*100
missing_percentage.sort_values(ascending=False)

In [ ]:
df['readmitted'].value_counts(normalize=True)*100

In [ ]:
# # Average hospital stay for each readmission group.
print(df.groupby("readmitted")["time_in_hospital"].mean())

# # Check if patients with more emergency visits are more likely to be readmitted.
print(df.groupby("readmitted")["number_emergency"].mean())

# Check if lab tests increase for high-risk patients
print(df.groupby("readmitted")["num_lab_procedures"].mean())

| Readmission | Avg Stay | Emergency Visits | Lab Tests |
| ----------- | -------- | ---------------- | --------- |
| **<30**     | 4.77     | 0.357            | 44.22     |
| **>30**     | 4.49     | 0.283            | 43.83     |
| **NO**      | 4.25     | 0.109            | 42.38     |


In [ ]:
df.groupby("age")["readmitted"].value_counts().unstack()

age_readmission = df.groupby("age")["readmitted"].value_counts(normalize=True).unstack()
age_readmission*100

| Age Group | <30       | >30     | NO      |
| --------- | --------- | ------- | ------- |
| 0–10      | 1.8%      | 16%     | **82%** |
| 10-20     | 5.8%      | 32.4%   | 62%     |
| 20–30     | **14%**   | 30%     | 55%     |
| 40–50     | 10.6%     | 33%     | 55%     |
| 60–70     | **11.1%** | 35%     | 53%     |
| 70–80     | **11.7%** | **36%** | 51%     |
| 80–90     | **12.0%** | **36%** | 51%     |


Patients aged 60–90 show consistently higher readmission rates, indicating increased vulnerability among elderly diabetic patients. However, an unexpected spike is observed in the 20–30 age group, suggesting potential lifestyle-related diabetes complications or poor treatment adherence among younger patients.

In [ ]:
# Insulin vs readmission rate(how insulin effect readmission rate)
insulin_readmission = df.groupby("insulin")["readmitted"].value_counts(normalize=True).unstack()
insulin_readmission*100

In [ ]:
print(" 1️⃣ Insulin vs Age")
print(pd.crosstab(df["age"], df["insulin"]))

print("2️⃣ Insulin vs Lab Tests")
print(df.groupby("insulin")["num_lab_procedures"].mean())

print("3️⃣ Insulin vs Hospital Stay")
print(df.groupby("insulin")["time_in_hospital"].mean())

In [ ]:
# finding relation between numeric coloum for better understanding

numeric_cols = [
"time_in_hospital",
"num_lab_procedures",
"num_procedures",
"num_medications",
"number_outpatient",
"number_emergency",
"number_inpatient",
"number_diagnoses"
]

df[numeric_cols].corr()

“Longer hospital stays increase the number of medications, procedures, and lab tests, which are indicators of patient severity. Patients with higher treatment intensity are more likely to be readmitted within 30 days. By monitoring these patients, hospitals can proactively intervene to reduce readmissions.”

In [ ]:
high_risk = df.groupby(['time_in_hospital', 'num_medications', 'num_procedures'])['readmitted'].value_counts(normalize=True).unstack()
high_risk.head(10)

In [69]:
df_seg = df.copy()

# Create simplified categorical features
df_seg['age_group'] = df_seg['age']
df_seg['insulin_status'] = df_seg['insulin']

# Select numeric features
numeric_features = ['time_in_hospital', 'num_medications', 'num_procedures']

# Function to compute readmission risk per patient
def compute_risk(row):
    # Basic risk scoring based on correlations we found
    risk = 0
    
    # Short stay (<3 days) but many meds/procedures increases risk
    if row['time_in_hospital'] <= 3:
        risk += 0.05 * row['num_medications']
        risk += 0.05 * row['num_procedures']
        
    # Longer stay (>4 days) adds more weight
    else:
        risk += 0.10 * row['time_in_hospital']
        risk += 0.05 * row['num_medications']
        risk += 0.05 * row['num_procedures']
    
    # Insulin up or down signals risk
    if row['insulin_status'] in ['Up', 'Down']:
        risk += 0.05
    
    return risk

# Apply risk computation
df_seg['risk_score'] = df_seg.apply(compute_risk, axis=1)

# Create high-risk flag
df_seg['high_risk_readmission'] = np.where(df_seg['risk_score'] >= 0.10, 1, 0)

# Select final columns for SQL / Power BI
final_table = df_seg[[
    'patient_nbr', 'age_group', 'insulin_status', 
    'time_in_hospital', 'num_medications', 'num_procedures', 
    'high_risk_readmission'
]]

# Check table
final_table.to_csv("C:\\Users\\abhay\\OneDrive\\Desktop\\Data-Analytics-Portfolio\\05_End-To-End-Project\\Healthcare-Readmission-Analysis\\Data\\diabetic_Cleaned_data.csv",index=False)